In [8]:
%load_ext autoreload
%autoreload 2

import deepxde as dde
import numpy as np
import torch
import matplotlib.pyplot as plt
from config import X_MIN_BURGERS, X_MAX_BURGERS, T_BURGERS
import zipfile, os

dde.config.set_default_float("float64")

for NU in [0.05]:#, 0.03, 0.02, 0.015, 0.01, 0.0075, 0.005]:
    print(f"\n{'='*50}")
    print(f"Entrenando PINN para nu={NU}")
    print(f"{'='*50}")

    # Setup — idéntico a tu celda 0
    geom = dde.geometry.Interval(X_MIN_BURGERS, X_MAX_BURGERS)
    timedomain = dde.geometry.TimeDomain(0, T_BURGERS)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)

    def pde_burgers(x, u):
        u_t  = dde.grad.jacobian(u, x, i=0, j=1)
        u_x  = dde.grad.jacobian(u, x, i=0, j=0)
        u_xx = dde.grad.hessian(u, x, i=0, j=0)
        return u_t + u * u_x - NU * u_xx

    def condicion_inicial_burgers(x):
        return -np.sin(np.pi * x[:, 0:1])

    ic = dde.icbc.IC(geomtime, condicion_inicial_burgers,
                     lambda x, on_initial: on_initial)
    bc_valores = dde.icbc.PeriodicBC(geomtime, component_x=0, component=0,
                                      on_boundary=lambda x, on_boundary: on_boundary)
    if False: 
        bc_derivadas = dde.icbc.PeriodicBC(geomtime, component_x=0, component=0,
                                        on_boundary=lambda x, on_boundary: on_boundary,
                                        derivative_order=1)

        data = dde.data.TimePDE(geomtime, pde_burgers, [ic, bc_valores, bc_derivadas],
                                num_domain=5000, num_initial=400, num_boundary=400)
    
    data = dde.data.TimePDE(geomtime, pde_burgers, [ic, bc_valores],
                                num_domain=5000, num_initial=400, num_boundary=400)

    red = dde.nn.FNN([2] + [50]*4 + [1], "silu", "Glorot uniform")
    model = dde.Model(data, red)

    # Adam — idéntico a tu celda 1
    model.compile("adam", lr=1e-3)
    losshistory, train_state = model.train(iterations=15000)

    # L-BFGS — idéntico a tu celda 2
    dde.optimizers.config.set_LBFGS_options(maxiter=10000, ftol=1e-14, gtol=1e-12)
    model.compile("L-BFGS")
    losshistory, train_state = model.train()

    if False:
        # Evaluación — idéntico a tu celda 3
        datos_ref = np.load(f'burgers_referencia_nu_super_fina{NU}.npz', mmap_mode='r')
        x_ref = datos_ref['x']
        u_ref_final = datos_ref['U'][-1]

    if True:
        with zipfile.ZipFile(f'burgers_referencia_nu_super_fina{NU}.npz') as zf:
            zf.extract('U.npy', '.')
            with zf.open('x.npy') as f:
                x_ref = np.load(f)

        U_mmap = np.load('U.npy', mmap_mode='r')  
        u_ref_final = np.array(U_mmap[-1])         # solo el último paso 
        del U_mmap
        import gc; gc.collect()
        try:
            os.remove('U.npy')
        except PermissionError:
            print("Aviso: no se pudo borrar U.npy, se borrará en la siguiente iteración")

    t_eval = np.full_like(x_ref, T_BURGERS)
    puntos_eval = np.vstack((x_ref, t_eval)).T
    u_pinn = model.predict(puntos_eval).flatten()

    error_l2_pinn = np.linalg.norm(u_pinn - u_ref_final) / np.linalg.norm(u_ref_final)
    print(f"Error L2 relativo (PINN vs referencia FDM fina): {error_l2_pinn:.6f}")

    # Guardado — idéntico a tu celda 5 pero siempre activo
    np.savez(f'burgers_pinn_resultado_nu{NU}_sin_derivadas.npz',
             x=x_ref, u_pinn=u_pinn, error_l2_pinn=error_l2_pinn,
             loss_steps=losshistory.steps,
             loss_total=np.sum(losshistory.loss_train, axis=1))

    torch.save({"model_state_dict": model.net.state_dict()},
               f"pinn_burgers_pesos_nu{NU}_sin_derivadas.pt")

    print(f"nu={NU} guardado correctamente.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Set the default float type to float64

Entrenando PINN para nu=0.05
Compiling model...
'compile' took 0.004317 s

Training model...

Step      Train loss                        Test loss                         Test metric
0         [2.61e-04, 5.01e-01, 3.71e-06]    [2.61e-04, 5.01e-01, 3.71e-06]    []  
1000      [5.57e-03, 2.07e-03, 4.54e-04]    [5.57e-03, 2.07e-03, 4.54e-04]    []  
2000      [1.82e-03, 7.01e-04, 1.16e-04]    [1.82e-03, 7.01e-04, 1.16e-04]    []  
3000      [8.50e-04, 2.19e-04, 4.67e-05]    [8.50e-04, 2.19e-04, 4.67e-05]    []  
4000      [5.11e-04, 1.10e-04, 2.37e-05]    [5.11e-04, 1.10e-04, 2.37e-05]    []  
5000      [3.27e-04, 6.30e-05, 1.34e-05]    [3.27e-04, 6.30e-05, 1.34e-05]    []  
6000      [2.15e-04, 4.38e-05, 7.53e-06]    [2.15e-04, 4.38e-05, 7.53e-06]    []  
7000      [1.63e-04, 4.81e-05, 1.96e-05]    [1.63e-04, 4.81e-05, 1.96e-05]    []  
8000      [1.18e-04, 2.78